## NBA 3.1 | Shared patient feature helpers (optimized)

Shared, validated builders for the claims feature notebooks (`RX/DX/PX/LAB_Patient_Features`). Load with `%run` **after** `00_event_list_config` and `00_cohort_generation`.

| Builder | Skill section | What changed vs. the old per-notebook code |
| --- | --- | --- |
| `build_first_occurrence` | 3 + 3.1 removal note | Same pre-aggregate -> pivot -> cross-join. Adds the NBA 3.1 censoring rule: events whose first claim is within 60 days of the data start for >50% of HCPs are **dropped**; events with <=10% are kept raw (recent launches); the rest are capped at 365 days + `{EVENT}_FIRST_CLAIM_BEFORE_WINDOW` flag. |
| `build_recency` | 4 + 4 removal note | One daily pre-aggregate and conditional aggregation (7 cohorts per pass) instead of 28 cohorts x 14 shuffles. Skewness from exact integer power sums. Unsuffixed totals never written. |
| `build_frequency` | 5 A-D + removal notes | No dense HCP x month spine, no 1,000+ column window. Rolling sums for each cohort snapshot month in one conditional aggregation; unique patients via a per-patient month **bitmask** (one pass instead of 28 x 5). Output = one row per HCP x cohort at `SVC_DT_M = LOOKBACK_END_DATE` (the row the Section 8.2 skeleton join uses). `LOGRATIO_FREQ_*` for N = 3, 6; `FREQ_*_IN_LAST_*` dropped when a `NUM_OF_PATIENTS_*` equivalent exists. |
| `build_hospitalization` | 6 | Same conditional aggregation + bitmask approach, snapshot-month grain. |

All small dimension tables (event map, 28-row cohort table, HCP universe) are broadcast. Intermediates are checkpointed to Delta (Section 7.4).

**Validation (2026-11-02, 300-HCP DX sample):** recency 237 columns, frequency 2,288 columns, hospitalization 9 columns and first occurrence 236 columns matched an independent pandas reference of the old logic exactly (0 mismatches).

In [0]:
import re, calendar, builtins   # builtins: 00_connection does `from pyspark.sql.functions import *` (shadows min/max)
from datetime import date
from functools import reduce
import pandas as pd
from pyspark.sql import functions as F

def sanitize_column_name(name):
    name = re.sub(r'[^A-Za-z0-9_]', '_', name.replace("+", "AND").replace("/", "_OR_"))
    name = re.sub(r'_+', '_', name).strip('_')
    return name.upper()

_COLUMN_MAP = {
    "PROVIDER_ID": ["PROVIDER_ID", "HCP_ID", "PHYSICIAN_ID", "PROV_ID", "BHID"],
    "PATIENT_ID":  ["PATIENT_ID", "MEMBER_ID", "ENROLLEE_ID", "SUBSCRIBER_ID"],
    "EVENT_NAME":  ["EVENT_NAME", "PRODUCT_GROUP", "DIAGNOSIS_DESCRIPTION", "PROCEDURE_DESCRIPTION", "TEST_TYPE"],
    "EVENT_DATE":  ["SVC_DT", "SERVICE_DATE", "CLAIM_DATE", "FILL_DT"],
    "CLAIM_ID":    ["CLAIM_ID", "CLAIM_NBR", "CLAIM_NUMBER"],
}

def standardize(df, event_type, extra_cols=()):
    """Canonical schema (Section 1.3). EVENT_DATE is always cast to DATE (PX SVC_DT is a string)."""
    for canonical, variants in _COLUMN_MAP.items():
        for v in variants:
            if v in df.columns and canonical not in df.columns:
                df = df.withColumnRenamed(v, canonical)
                break
    df = df.withColumn("EVENT_DATE", F.to_date("EVENT_DATE")).withColumn("EVENT_TYPE", F.lit(event_type))
    keep = [c for c in ["PROVIDER_ID", "PATIENT_ID", "EVENT_NAME", "EVENT_DATE", "EVENT_TYPE",
                        "CLAIM_ID", "DATA_SOURCE", *extra_cols] if c in df.columns]
    return df.select(*keep)

def _to_date(x):
    return pd.Timestamp(x).date()

def _add_months(d, k):
    y, m = divmod(d.month - 1 + k, 12)
    return date(d.year + y, m + 1, 1)

def _last_day(d):
    return date(d.year, d.month, calendar.monthrange(d.year, d.month)[1])

def _month_idx(d, base):
    return (d.year - base.year) * 12 + (d.month - base.month)

def cohort_rows(cohorts):
    """28 cohorts as python dicts. END = 1st of the last INCLUDED month (Section 10.8)."""
    rows = []
    for _, r in cohorts.iterrows():
        s, e = _to_date(r["lookback_start_date"]), _to_date(r["lookback_end_date"])
        rows.append(dict(COHORT=r["cohort_id"], START=s, END=e,
                         END_EXCL=_add_months(e, 1), LAST_DAY=_last_day(e)))
    base = builtins.min(r["START"] for r in rows)
    for r in rows:
        r["S_IDX"], r["E_IDX"] = _month_idx(r["START"], base), _month_idx(r["END"], base)
    return rows, base

def _anchor_df(rows):
    """Small (28-row) cohort table: always broadcast it."""
    return F.broadcast(spark.createDataFrame(
        [(r["COHORT"], r["END"]) for r in rows], "COHORT string, LOOKBACK_END_DATE date"))

def event_map_df(event_keys):
    san = {e: sanitize_column_name(e) for e in event_keys}
    assert len(set(san.values())) == len(san), "sanitize_column_name collision in event keys"
    return F.broadcast(spark.createDataFrame(list(san.items()), "EVENT_NAME string, EVENT_SAN string")), san

def _checkpoint(df, path):
    """Write an intermediate to Delta and read it back (Section 7.4). Works on serverless and classic."""
    if path is None:
        return df
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(path)
    return spark.read.format("delta").load(path)

def _explode_cohorts(df, keys, rows, fields):
    """Wide (one column per cohort x field, named f'{cohort}__{field}') -> long (one row per cohort)."""
    structs = [F.struct(F.lit(r["COHORT"]).alias("COHORT"),
                        *[F.col(f"{r['COHORT']}__{f}").alias(f) for f in fields]) for r in rows]
    return df.select(*keys, F.explode(F.array(*structs)).alias("_x")).select(*keys, "_x.*")

def _chunks(xs, n):
    return [xs[i:i + n] for i in range(0, len(xs), n)]

def _window_mask(r, base, w):
    """Bitmask of month indexes inside the cohort lookback whose last day is within w days of the
    last day of the LOOKBACK_END month (same rule as the old DAYS_DIFF <= w on SVC_DT_M)."""
    mask, m = 0, r["START"]
    while m <= r["END"]:
        if (r["LAST_DAY"] - _last_day(m)).days <= w:
            mask |= (1 << _month_idx(m, base))
        m = _add_months(m, 1)
    return mask

def _patient_counts(claims_m, rows, base, windows, group_keys, name_fn, event_keys_san=None):
    """Distinct patients per (group_keys, cohort, window) in ONE pass: bit_or of the patient's
    active months, then a bit test per cohort/window. Replaces 28 x len(windows) filter+dedup scans."""
    max_idx = builtins.max(r["E_IDX"] for r in rows)
    assert max_idx < 63, "month bitmask needs < 63 months"
    pm = (claims_m.filter(F.col("PATIENT_ID").isNotNull() & F.col("M_IDX").between(0, max_idx))
                  .groupBy(*group_keys, "PATIENT_ID")
                  .agg(F.expr("bit_or(shiftleft(1L, M_IDX))").alias("MASK")))
    aggs = [F.sum(F.when((F.col("MASK").bitwiseAND(F.lit(_window_mask(r, base, w)))) != 0, 1)).alias(f"{r['COHORT']}__P{w}")
            for r in rows for w in windows]
    g = pm.groupBy(*group_keys).agg(*aggs)
    long = _explode_cohorts(g, group_keys, rows, [f"P{w}" for w in windows]).filter(F.col(f"P{builtins.max(windows)}").isNotNull())
    if event_keys_san is None:
        return long.select("PROVIDER_ID", "COHORT", *[F.col(f"P{w}").alias(name_fn(None, w)) for w in windows])
    piv = (long.groupBy("PROVIDER_ID", "COHORT").pivot("EVENT_SAN", event_keys_san)
               .agg(*[F.first(f"P{w}").alias(f"P{w}") for w in windows]))
    return piv.select("PROVIDER_ID", "COHORT",
                      *[F.col(f"{s}_P{w}").alias(name_fn(s, w)) for s in event_keys_san for w in windows])

In [0]:
def build_first_occurrence(claims, event_keys, cohorts, tmp_path=None, apply_censoring=True,
                           censor_window_days=60, censor_drop_share=0.50, recent_launch_max_share=0.10,
                           cap_days=365):
    """Pre-aggregate -> pivot -> cross-join cohorts (Section 3) + NBA 3.1 censoring rules.
    Share of HCPs whose first claim is within censor_window_days of the data start:
      > censor_drop_share        -> column DROPPED (left-censored, measures the data start)
      <= recent_launch_max_share -> raw value kept (launched inside the data window)
      otherwise                  -> min(days, cap_days) + {EVENT}_FIRST_CLAIM_BEFORE_WINDOW flag
    Returns (features_df, censor_report_pandas)."""
    rows, _ = cohort_rows(cohorts)
    evmap, san = event_map_df(event_keys)
    pre = _checkpoint(claims.join(evmap, "EVENT_NAME")
                            .groupBy("PROVIDER_ID", "EVENT_SAN")
                            .agg(F.min("EVENT_DATE").alias("FIRST_OCCURRENCE_DATE")), tmp_path)
    data_start = pre.agg(F.min("FIRST_OCCURRENCE_DATE")).first()[0]
    rep = (pre.groupBy("EVENT_SAN")
              .agg(F.count(F.lit(1)).alias("N_HCP"),
                   F.avg((F.datediff("FIRST_OCCURRENCE_DATE", F.lit(data_start)) <= censor_window_days)
                         .cast("double")).alias("SHARE_NEAR_DATA_START"))
              .toPandas())
    def _rule(s):
        if not apply_censoring: return "KEEP"
        if s > censor_drop_share: return "DROP_CENSORED"
        if s <= recent_launch_max_share: return "KEEP"
        return "CAP"
    rep["RULE"] = rep["SHARE_NEAR_DATA_START"].apply(_rule)
    rep["DATA_START"] = data_start
    rule = dict(zip(rep["EVENT_SAN"], rep["RULE"]))
    kept = [s for s in sorted(san.values()) if rule.get(s, "KEEP") != "DROP_CENSORED"]

    wide = pre.groupBy("PROVIDER_ID").pivot("EVENT_SAN", kept).agg(F.min("FIRST_OCCURRENCE_DATE"))
    fo = wide.crossJoin(_anchor_df(rows))
    cutoff = F.last_day(F.col("LOOKBACK_END_DATE"))
    out_cols, val_cols = [], []
    for s in kept:
        days = F.datediff(cutoff, F.col(s))
        val = F.when(days >= 0, days)                       # future-leakage guard
        name = f"MAX_EVENT_TIME_{s}_FIRST_OCCURRENCE"
        if rule.get(s) == "CAP":
            out_cols += [F.when(val.isNotNull(), F.least(val, F.lit(cap_days))).alias(name),   # least() skips NULLs
                         F.when(val.isNotNull(), (val > cap_days).cast("int")).alias(f"{s}_FIRST_CLAIM_BEFORE_WINDOW")]
        else:
            out_cols.append(val.alias(name))
        val_cols.append(name)
    fo = fo.select("PROVIDER_ID", "COHORT", "LOOKBACK_END_DATE", *out_cols)
    fo = fo.filter(F.greatest(*[F.col(c).isNotNull().cast("int") for c in val_cols]) == 1)
    fo = fo.withColumn("HCP_COHORT_ID", F.concat_ws("_", "PROVIDER_ID", "COHORT"))
    return fo, rep.sort_values("SHARE_NEAR_DATA_START", ascending=False).reset_index(drop=True)

In [0]:
def build_recency(claims, event_keys, cohorts, suffix, windows=(30, 60, 90, 180, 270, 360),
                  count_claim_ids=True, cohort_batch=7, tmp_base=None):
    """Same outputs as the old 28-cohort loop, from one daily pre-aggregate:
    claims -> (PROVIDER_ID, EVENT, EVENT_DATE, N) -> per cohort conditional min/sum/power sums
    (cohort_batch cohorts per pass) -> skewness from exact integer power sums."""
    rows, _ = cohort_rows(cohorts)
    evmap, san = event_map_df(event_keys)
    lo, hi = builtins.min(r["START"] for r in rows), builtins.max(r["END_EXCL"] for r in rows)
    n_clm = F.count("CLAIM_ID") if count_claim_ids else F.count(F.lit(1))
    daily = _checkpoint(claims.filter((F.col("EVENT_DATE") >= F.lit(lo)) & (F.col("EVENT_DATE") < F.lit(hi)))
                              .join(evmap, "EVENT_NAME")
                              .groupBy("PROVIDER_ID", "EVENT_SAN", "EVENT_DATE")
                              .agg(F.count(F.lit(1)).alias("N_ALL"), n_clm.alias("N_CLM")),
                        f"{tmp_base}/_rec_daily" if tmp_base else None)
    fields = ["MIN_T", "N_CLM"] + [f"{k}_{w}" for w in windows for k in ("S0", "NC", "S1", "S2", "S3")]
    parts = []
    for batch in _chunks(rows, cohort_batch):
        aggs = []
        for r in batch:
            c = r["COHORT"]
            inw = (F.col("EVENT_DATE") >= F.lit(r["START"])) & (F.col("EVENT_DATE") < F.lit(r["END_EXCL"]))
            t = F.datediff(F.lit(r["LAST_DAY"]), F.col("EVENT_DATE")).cast("long")
            aggs += [F.min(F.when(inw, t)).alias(f"{c}__MIN_T"), F.sum(F.when(inw, F.col("N_CLM"))).alias(f"{c}__N_CLM")]
            for w in windows:
                cw = inw & (t <= w)
                aggs += [F.sum(F.when(cw, F.col("N_ALL"))).alias(f"{c}__S0_{w}"),
                         F.sum(F.when(cw, F.col("N_CLM"))).alias(f"{c}__NC_{w}"),
                         F.sum(F.when(cw, F.col("N_ALL") * t)).alias(f"{c}__S1_{w}"),
                         F.sum(F.when(cw, F.col("N_ALL") * t * t)).alias(f"{c}__S2_{w}"),
                         F.sum(F.when(cw, F.col("N_ALL") * t * t * t)).alias(f"{c}__S3_{w}")]
        g = daily.groupBy("PROVIDER_ID", "EVENT_SAN").agg(*aggs)
        parts.append(_explode_cohorts(g, ["PROVIDER_ID", "EVENT_SAN"], batch, fields).filter(F.col("MIN_T").isNotNull()))
    long = _checkpoint(reduce(lambda a, b: a.unionByName(b), parts),
                       f"{tmp_base}/_rec_long" if tmp_base else None)

    hcp_aggs = [F.sum("N_CLM").alias(f"TOTAL_EVENTS_{suffix}"), F.count(F.lit(1)).alias(f"UNIQUE_EVENTS_{suffix}")]
    for w in windows:
        hcp_aggs += [F.when(F.sum(f"S0_{w}") > 0, F.sum(f"NC_{w}")).alias(f"TOTAL_EVENTS_TIME_{w}_DAYS_{suffix}"),
                     F.sum(F.when(F.col("MIN_T") <= w, 1)).alias(f"UNIQUE_EVENTS_TIME_{w}_DAYS_{suffix}"),
                     *[F.sum(f"{k}_{w}").alias(f"_{k}_{w}") for k in ("S0", "S1", "S2", "S3")]]
    hcp = long.groupBy("PROVIDER_ID", "COHORT").agg(*hcp_aggs)
    sk = []
    for w in windows:
        n, s1, s2, s3 = [F.col(f"_{k}_{w}").cast("double") for k in ("S0", "S1", "S2", "S3")]
        mean = s1 / n
        m2 = s2 / n - mean * mean
        m3 = s3 / n - 3 * mean * s2 / n + 2 * mean * mean * mean
        sk.append(F.when((n > 0) & (m2 > 1e-9), m3 / F.pow(m2, 1.5)).alias(f"SKEWNESS_TIME_{w}_DAYS_{suffix}"))
    hcp = hcp.select("PROVIDER_ID", "COHORT", *[c for c in hcp.columns if not c.startswith("_") and c not in ("PROVIDER_ID", "COHORT")], *sk)

    sans = sorted(san.values())
    sub1 = (long.groupBy("PROVIDER_ID", "COHORT").pivot("EVENT_SAN", sans).agg(F.min("MIN_T"))
                .select("PROVIDER_ID", "COHORT", *[F.col(s).alias(f"MIN_EVENT_TIME_{s}_TIME_MIN") for s in sans]))
    rec = (hcp.join(sub1, ["PROVIDER_ID", "COHORT"], "left")
              .withColumn("HCP_COHORT_ID", F.concat_ws("_", "PROVIDER_ID", "COHORT")))
    # NBA 3.1 removal note (Section 4): never keep unsuffixed TOTAL_/UNIQUE_ duplicates
    unsuffixed = [c for c in rec.columns if c.startswith(("TOTAL_", "UNIQUE_"))
                  and not c.endswith(("_RX", "_DX", "_PX", "_LAB_DATA"))]
    return rec.drop(*unsuffixed)

In [0]:
def build_frequency(claims_freq, event_keys, cohorts, patient_prefix, lag_windows=(1, 2, 3, 6, 12),
                    change_windows=(3, 6), patient_windows=(90, 180, 360),
                    drop_freq_with_patient_equiv=True, tmp_base=None):
    """claims_freq: PROVIDER_ID, PATIENT_ID, EVENT_NAME, SVC_DT_M.
    Output grain: one row per PROVIDER_ID x COHORT at the cohort snapshot month
    (SVC_DT_M = LOOKBACK_END_DATE), the row the skeleton join in Section 8.2 uses.
    No dense spine, no 1,000+ column window, no 28 x 5 patient scans."""
    rows, base = cohort_rows(cohorts)
    evmap, san = event_map_df(event_keys)
    sans = sorted(san.values())
    cm = (claims_freq.join(evmap, "EVENT_NAME")
                     .withColumn("M_IDX", F.months_between(F.trunc("SVC_DT_M", "month"), F.lit(base)).cast("int")))
    cm = _checkpoint(cm.select("PROVIDER_ID", "PATIENT_ID", "EVENT_SAN", "M_IDX"),
                     f"{tmp_base}/_freq_claims_m" if tmp_base else None)

    # Sub-A: monthly counts (long format, not pivoted)
    monthly = cm.groupBy("PROVIDER_ID", "EVENT_SAN", "M_IDX").agg(F.count(F.lit(1)).alias("N"))
    monthly = _checkpoint(monthly, f"{tmp_base}/_freq_monthly" if tmp_base else None)

    # Sub-B: rolling sums at each cohort snapshot month in one pass (conditional sums)
    need = sorted(set(change_windows) | {12} | (set() if drop_freq_with_patient_equiv else set(lag_windows)))
    aggs = [F.sum(F.when(F.col("M_IDX").between(r["E_IDX"] - (n - 1), r["E_IDX"]), F.col("N"))).alias(f"{r['COHORT']}__F{n}")
            for r in rows for n in need]
    g = monthly.groupBy("PROVIDER_ID", "EVENT_SAN").agg(*aggs)
    long = _explode_cohorts(g, ["PROVIDER_ID", "EVENT_SAN"], rows, [f"F{n}" for n in need]).filter(F.col("F12").isNotNull())
    # Sub-C: NBA 3.1 log-ratio (N = 3, 6 only)
    f12 = F.coalesce(F.col("F12"), F.lit(0))
    for n in change_windows:
        fn = F.coalesce(F.col(f"F{n}"), F.lit(0))
        long = long.withColumn(f"LR{n}", F.log(((fn / n) + 0.5) / (((f12 - fn) / (12 - n)) + 0.5)))
    out_fields = [f"LR{n}" for n in change_windows] + ([] if drop_freq_with_patient_equiv else [f"F{n}" for n in lag_windows])
    piv = (long.groupBy("PROVIDER_ID", "COHORT").pivot("EVENT_SAN", sans)
               .agg(*[F.first(f).alias(f) for f in out_fields]))
    def _name(s, f):
        return (f"LOGRATIO_FREQ_{s}_IN_LAST_{f[2:]}_MONTH" if f.startswith("LR")
                else f"FREQ_{s}_IN_LAST_{f[1:]}_MONTH")
    piv = piv.select("PROVIDER_ID", "COHORT", *[F.col(f"{s}_{f}").alias(_name(s, f)) for s in sans for f in out_fields])

    # skeleton = HCPs with any claim x 28 cohorts (same row set as the old dense spine at the snapshot month)
    skeleton = monthly.select("PROVIDER_ID").distinct().crossJoin(_anchor_df(rows)).withColumnRenamed("LOOKBACK_END_DATE", "SVC_DT_M")
    freq = skeleton.join(piv, ["PROVIDER_ID", "COHORT"], "left")
    freq = freq.fillna(0, subset=[c for c in freq.columns if c.startswith(("FREQ_", "LOGRATIO_FREQ_"))])

    # Sub-D: unique patients (bitmask, one pass)
    sub_d = _patient_counts(cm, rows, base, patient_windows, ["PROVIDER_ID", "EVENT_SAN"],
                            lambda s, w: f"NUM_OF_PATIENTS_THE_HCP_{patient_prefix}_{s}_IN_LAST_{w}_DAYS", sans)
    return (freq.join(sub_d, ["PROVIDER_ID", "COHORT"], "left")
                .withColumn("HCP_COHORT_ID", F.concat_ws("_", "PROVIDER_ID", "COHORT")))

In [0]:
def build_hospitalization(total_freq, hosp_freq, cohorts, collapsed, patient_label,
                          lag_windows=(1, 6, 12), patient_windows=(30, 180, 360)):
    """total_freq / hosp_freq: PROVIDER_ID, PATIENT_ID, SVC_DT_M. Snapshot-month grain like build_frequency."""
    rows, base = cohort_rows(cohorts)
    def _m(df): return df.withColumn("M_IDX", F.months_between(F.trunc("SVC_DT_M", "month"), F.lit(base)).cast("int"))
    tot = _m(total_freq).groupBy("PROVIDER_ID", "M_IDX").agg(F.count(F.lit(1)).alias("T"))
    hosp_m = _m(hosp_freq)
    hos = hosp_m.groupBy("PROVIDER_ID", "M_IDX").agg(F.count(F.lit(1)).alias("H"))
    a = tot.join(hos, ["PROVIDER_ID", "M_IDX"], "left").fillna(0, subset=["H"])   # same as the old left join
    aggs = []
    for r in rows:
        for n in lag_windows:
            inw = F.col("M_IDX").between(r["E_IDX"] - (n - 1), r["E_IDX"])
            aggs += [F.sum(F.when(inw, F.col("T"))).alias(f"{r['COHORT']}__T{n}"),
                     F.sum(F.when(inw, F.col("H"))).alias(f"{r['COHORT']}__H{n}")]
    g = a.groupBy("PROVIDER_ID").agg(*aggs)
    long = _explode_cohorts(g, ["PROVIDER_ID"], rows, [f"{k}{n}" for n in lag_windows for k in ("T", "H")])
    long = long.fillna(0, subset=[f"{k}{n}" for n in lag_windows for k in ("T", "H")])
    cond = reduce(lambda x, y: x & y, [F.col(f"T{n}") >= F.col(f"H{n}") for n in lag_windows])   # validation filter
    long = long.filter(cond)
    out = long.select("PROVIDER_ID", "COHORT",
                      *[F.col(f"H{n}").alias(f"TOTAL_HOSPITALIZED_CLAIMS_{collapsed}_IN_LAST_{n}_MONTH") for n in lag_windows],
                      *[F.when(F.col(f"T{n}") != 0, F.col(f"H{n}") / F.col(f"T{n}")).alias(
                          f"PERC_HOSPITALIZED_HCP_CLAIMS_FOR_{collapsed}_IN_LAST_{n}_MONTH") for n in lag_windows])
    anchor = _anchor_df(rows).withColumnRenamed("LOOKBACK_END_DATE", "SVC_DT_M")
    out = out.join(anchor, "COHORT")
    sub_d = _patient_counts(hosp_m, rows, base, patient_windows, ["PROVIDER_ID"],
                            lambda s, w: f"NUM_OF_HOSPITALIZED_PATIENTS_THE_HCP_{patient_label}_IN_LAST_{w}_DAYS")
    return (out.join(sub_d, ["PROVIDER_ID", "COHORT"], "left")
               .withColumn("HCP_COHORT_ID", F.concat_ws("_", "PROVIDER_ID", "COHORT")))

print("Patient feature helpers loaded: build_first_occurrence, build_recency, build_frequency, build_hospitalization")